In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Tüm sütunların ve satırların rahatça görünmesi için ayarlar
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: '%.3f' % x)


In [5]:
import os

print("Çalışma Dizini:", os.getcwd())
print("Dizindeki Dosyalar/Klasörler:", os.listdir())

Çalışma Dizini: C:\Users\USER\ecommerce-intelligence\notebooks
Dizindeki Dosyalar/Klasörler: ['.ipynb_checkpoints', '1_data_exploration.ipynb']


In [7]:
import os

target_dir = os.path.abspath("../data/raw")
os.makedirs(target_dir, exist_ok=True)

print("Klasör hazırlandı:", target_dir)

Klasör hazırlandı: C:\Users\USER\ecommerce-intelligence\data\raw


In [8]:
import os
os.system(f'explorer "{target_dir}"')

1

In [9]:
import os
import pandas as pd

raw_files = [f for f in os.listdir(target_dir) if not f.startswith('.')]
print("raw klasöründeki dosyalar:", raw_files)

# Excel dosyasını seçip yükle
excel_file = [f for f in raw_files if f.endswith(('.xlsx', '.xls', '.csv'))][0]
file_path = os.path.join(target_dir, excel_file)
print(f"Okunan dosya: {file_path}")

if excel_file.endswith('.csv'):
    df = pd.read_csv(file_path)
else:
    df = pd.read_excel(file_path, sheet_name='Year 2009-2010')

df_raw = df.copy()
df.head()

raw klasöründeki dosyalar: ['online_retail_II.xlsx']
Okunan dosya: C:\Users\USER\ecommerce-intelligence\data\raw\online_retail_II.xlsx


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.950,13085.000,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.750,13085.000,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.750,13085.000,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.100,13085.000,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.250,13085.000,United Kingdom


In [10]:
print(f"Toplam Satır Sayısı: {df.shape[0]}")
print(f"Toplam Sütun Sayısı: {df.shape[1]}")
print("\n--- Veri Tipleri ve Bellek Kullanımı ---")
df.info()

Toplam Satır Sayısı: 525461
Toplam Sütun Sayısı: 8

--- Veri Tipleri ve Bellek Kullanımı ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [11]:
null_counts = df.isnull().sum()
null_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Eksik Sayısı': null_counts, 'Yüzde (%)': null_percent})
missing_df[missing_df['Eksik Sayısı'] > 0]

,Eksik Sayısı,Yüzde (%)
Description,2928,0.557
Customer ID,107927,20.539


In [12]:
df.describe([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T

,count,mean,min,1%,5%,25%,50%,75%,95%,99%,max,std
Quantity,525461.000,10.338,-9600.000,-3.000,1.000,1.000,3.000,10.000,30.000,120.000,19152.000,107.424
InvoiceDate,525461,2010-06-28 11:37:36.845017856,2009-12-01 07:45:00,2009-12-02 14:36:00,2009-12-13 10:48:00,2010-03-21 12:20:00,2010-07-06 09:51:00,2010-10-15 12:45:00,2010-11-29 15:18:00,2010-12-08 10:40:00,2010-12-09 20:01:00,NaN
Price,525461.000,4.689,-53594.360,0.210,0.420,1.250,2.100,4.210,10.170,19.950,25111.090,146.127
Customer ID,417534.000,15360.645,12346.000,12435.000,12725.000,13983.000,15311.000,16799.000,17913.000,18196.000,18287.000,1680.811


In [13]:
# Fatura numarası metne çevrilip başında 'C' olan satırları inceliyoruz
c_invoices = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"İptal Fatura Satır Sayısı: {len(c_invoices)}")
print(f"İptal Faturaların Toplam Veriye Oranı: %{(len(c_invoices) / len(df)) * 100:.2f}")

İptal Fatura Satır Sayısı: 10206
İptal Faturaların Toplam Veriye Oranı: %1.94


In [14]:
customer_col = 'Customer ID' if 'Customer ID' in df.columns else 'CustomerID'

print(f"Benzersiz Fatura Sayısı: {df['Invoice'].nunique()}")
print(f"Benzersiz Ürün Sayısı: {df['StockCode'].nunique()}")
print(f"Benzersiz Müşteri Sayısı: {df[customer_col].nunique()}")

Benzersiz Fatura Sayısı: 28816
Benzersiz Ürün Sayısı: 4632
Benzersiz Müşteri Sayısı: 4383


In [15]:
def clean_data(dataframe):
    df_clean = dataframe.copy()
    
    # 1. Eksik Customer ID ve Description satırlarını çıkar
    df_clean.dropna(subset=['Customer ID', 'Description'], inplace=True)
    
    # 2. İptal edilen faturaları (C ile başlayanlar) filtrele
    df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]
    
    # 3. Negatif ve sıfır Quantity/Price değerlerini temizle
    df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]
    
    # 4. Customer ID tipini float'tan int'e (tamsayıya) çevir
    df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)
    
    # 5. Toplam Harcama (TotalPrice) sütununu ekle: Quantity * Price
    df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['Price']
    
    return df_clean

# Temizleme fonksiyonunu çalıştır
df_cleaned = clean_data(df_raw)

print(f"Ham Veri Satır Sayısı   : {df_raw.shape[0]}")
print(f"Temiz Veri Satır Sayısı : {df_cleaned.shape[0]}")
print(f"Temiz Verideki Benzersiz Müşteri: {df_cleaned['Customer ID'].nunique()}")
print(f"Temiz Verideki Benzersiz Fatura : {df_cleaned['Invoice'].nunique()}")

Ham Veri Satır Sayısı   : 525461
Temiz Veri Satır Sayısı : 407664
Temiz Verideki Benzersiz Müşteri: 4312
Temiz Verideki Benzersiz Fatura : 19213


In [16]:
import os

processed_dir = os.path.abspath("../data/processed")
os.makedirs(processed_dir, exist_ok=True)

# Sonraki günlerde hızlıca okumak için CSV formatında kaydediyoruz
output_path = os.path.join(processed_dir, "cleaned_retail.csv")
df_cleaned.to_csv(output_path, index=False)
print(f"Temizlenmiş veri başarıyla kaydedildi: {output_path}")

Temizlenmiş veri başarıyla kaydedildi: C:\Users\USER\ecommerce-intelligence\data\processed\cleaned_retail.csv
